# Curvature-dependent normal stress - AVAC

This notebook verifies the Fischer et al. (2012) centripetal correction to Coulomb normal stress for planar, concave-circular, and convex-circular tracks. It then isolates the changing-basis term that a terrain-following material point acquires on a curved track but AVAC's deliberately reduced local Cartesian source omits.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('AVAC', 'Curvature_normal_stress')
CORES = max(1, os.cpu_count() or 1)
case.path


## Controlled source definition

The first suite places a one-metre-deep material element moving at 8 m s$^{-1}$ at each track nadir. With radius 40 m and $\mu=0.30$, the compiled Fortran source is compared with $\mathrm{d}u/\mathrm{d}t=-\mu(g+\kappa u^2)$ for $\kappa=0$, $+1/R$, and $-1/R$. The convex case remains below loss of contact, and one source step is compared with eight substeps. A frozen 30-degree tangent with zero applied force also checks that the reduced scalar source does not invent map-plane acceleration.


In [ ]:
case.run('run_curvature_validation.py')


## Curvature-source agreement


In [ ]:
source_summary = case.json('results/summary.json')
source_summary


In [ ]:
case.show('figures/curvature_source_verification.png')


## Terrain-following circular-track diagnostic

The frozen-cell check above is not a terrain-following transport test. This second executable integrates squared surface speed $w=v_s^2$ along a circular transition of radius 400 m from a 34-degree slope to horizontal. It compares constrained terrain-following point dynamics with the surface-speed equation implied by AVAC's reduced horizontal-velocity projection. Flat and constant-slope Coulomb tracks are exact controls; a zero-force circle isolates the changing projection analytically. The exercise excludes depth and pressure evolution, so it documents a coordinate discrepancy and does not prescribe a production source correction.


In [ ]:
case.run('run_circular_track_verification.py')


In [ ]:
circular_summary = case.json('results/circular_track_summary.json')
circular_summary


In [ ]:
case.show('figures/circular_track_coordinate_verification.png')
